# Cleanup — Delete All Demo Resources

This script **deletes every resource** created by `02_Lakebase_Setup` and
`04_App_Deployment`. Run cells sequentially from top to bottom.

All operations use the **Databricks Python SDK** and work on **serverless compute**.
Every step is idempotent — safe to re-run if something was already deleted.

### What gets deleted

| # | Resource | Name / Location |
|---|----------|----------------|
| 1 | Databricks App | `delivery-slot-booking` (app + service principal) |
| 2 | Synced tables | `ekko`, `ekpo_enriched` (Lakebase sync jobs) |
| 3 | Lakebase project | `delivery-slot-booking` (cascades: branches, endpoints, databases, data) |
| 4 | Delta tables | `ekko`, `ekpo`, `ekpo_enriched`, `dock_slot`, `delivery_booking` |
| 5 | Unity Catalog schema | `{CATALOG}.{SCHEMA}` |

### What is preserved

* The Unity Catalog **catalog** itself
* Your compute and authentication
* This repository and all notebooks
* The `app/` source code directory (only the deployment copy at `apps/` is recreated on each deploy)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, BadRequest
import shutil, os, time

w = WorkspaceClient()

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Update these if you used different names during setup.

APP_NAME = "delivery-slot-booking"
PROJECT = "delivery-slot-booking"

CATALOG = "classic_stable_4rp118_catalog"
SCHEMA = "delivery_slot_booking_ppmaxkohler"
FULL_SCHEMA = f"{CATALOG}.{SCHEMA}"

USER = w.current_user.me().user_name
WORKSPACE_APP_PATH = f"/Workspace/Users/{USER}/apps/{APP_NAME}"

print(f"App name:         {APP_NAME}")
print(f"Lakebase project: {PROJECT}")
print(f"Delta schema:     {FULL_SCHEMA}")
print(f"User:             {USER}")
print(f"App files:        {WORKSPACE_APP_PATH}")
print()
print("⚠️  Running this notebook will PERMANENTLY DELETE all of the above.")

## Step 1: Delete the Databricks App

Stops the app, deletes its compute, and removes the associated service principal.

In [0]:
print("Step 1: Deleting Databricks App...")
try:
    w.api_client.do("DELETE", f"/api/2.0/apps/{APP_NAME}")
    print(f"  ✓ App '{APP_NAME}' delete initiated.")

    # Wait for deletion to complete
    for i in range(20):
        try:
            info = w.api_client.do("GET", f"/api/2.0/apps/{APP_NAME}")
            state = info.get("app_status", {}).get("state", "UNKNOWN")
            print(f"    Waiting... state={state}")
            time.sleep(10)
        except NotFound:
            print(f"  ✓ App '{APP_NAME}' fully deleted.")
            break
    else:
        print(f"  ⚠ App deletion still in progress after 200s — check manually.")

except NotFound:
    print(f"  ⊘ App '{APP_NAME}' does not exist — skipping.")
except Exception as e:
    print(f"  ⚠ Error: {e}")

## Step 2: Delete Synced Tables

Removes the Delta-to-Lakebase sync jobs. Must be done **before** deleting the
Lakebase project, otherwise the sync jobs become orphaned.

In [0]:
print("Step 2: Deleting synced tables...")

synced_tables = [
    f"{CATALOG}.{SCHEMA}.ekko",
    f"{CATALOG}.{SCHEMA}.ekpo_enriched",
]

for table_fqn in synced_tables:
    try:
        w.api_client.do("DELETE", f"/api/2.0/postgres/synced_tables/{table_fqn}")
        print(f"  \u2713 Synced table '{table_fqn}' deleted.")
    except NotFound:
        print(f"  \u2298 Synced table '{table_fqn}' not found \u2014 skipping.")
    except BadRequest as e:
        if "Unsupported catalog kind" in str(e):
            print(f"  \u2298 '{table_fqn}' is not a synced table \u2014 skipping.")
        else:
            print(f"  \u26a0 Error deleting '{table_fqn}': {e}")
    except Exception as e:
        print(f"  \u26a0 Error deleting '{table_fqn}': {e}")

## Step 3: Delete the Lakebase Project

Deleting the project **cascades** to all branches (`production`, `dev`),
endpoints (`primary`, `read-write`), databases (`delivery_app`), and all
Postgres data. This is irreversible.

In [0]:
print("Step 3: Deleting Lakebase project...")

# Unprotect production branch first (required for deletion)
try:
    w.api_client.do(
        "PATCH",
        f"/api/2.0/postgres/projects/{PROJECT}/branches/production",
        body={"spec": {"is_protected": False}},
        query={"update_mask": "spec.is_protected"}
    )
    print("  Unprotected production branch.")
except Exception:
    pass  # Branch may not exist or already unprotected

# Delete the project
try:
    w.api_client.do("DELETE", f"/api/2.0/postgres/projects/{PROJECT}")
    print(f"  ✓ Project '{PROJECT}' delete initiated.")

    # Wait for deletion
    for i in range(20):
        try:
            w.api_client.do("GET", f"/api/2.0/postgres/projects/{PROJECT}")
            print(f"    Waiting for project deletion... ({(i+1)*10}s)")
            time.sleep(10)
        except NotFound:
            print(f"  ✓ Project '{PROJECT}' fully deleted.")
            break
    else:
        print(f"  ⚠ Project deletion still in progress — check manually.")

except NotFound:
    print(f"  ⊘ Project '{PROJECT}' does not exist — skipping.")
except Exception as e:
    print(f"  ⚠ Error: {e}")

## Step 4: Drop Delta Tables and Schema

Drops all demo tables from Unity Catalog, then drops the schema.
The catalog itself is preserved.

In [0]:
TABLES = ["ekko", "ekpo", "ekpo_enriched", "dock_slot", "delivery_booking"]

print(f"Step 4: Dropping tables from {FULL_SCHEMA}...")
for table in TABLES:
    full_name = f"{FULL_SCHEMA}.{table}"
    try:
        spark.sql(f"DROP TABLE IF EXISTS {full_name}")
        print(f"  ✓ Dropped {full_name}")
    except Exception as e:
        print(f"  ⚠ Could not drop {full_name}: {e}")

print(f"\nDropping schema {FULL_SCHEMA}...")
try:
    spark.sql(f"DROP SCHEMA IF EXISTS {FULL_SCHEMA} CASCADE")
    print(f"  ✓ Schema '{FULL_SCHEMA}' dropped.")
except Exception as e:
    print(f"  ⚠ Could not drop schema: {e}")

## Summary

If all steps above show ✓, the following have been cleaned up:

| Resource | Status |
|----------|--------|
| Databricks App (`delivery-slot-booking`) | Deleted |
| Synced tables (ekko, ekpo_enriched) | Deleted |
| Lakebase project (`delivery-slot-booking`) | Deleted (all branches, endpoints, data) |
| Delta tables (ekko, ekpo, ekpo_enriched, dock_slot, delivery_booking) | Dropped |
| Schema (`{CATALOG}.{SCHEMA}`) | Dropped |

To re-create everything from scratch, run:
1. `01_SAP_Data_Pipeline` → generate SAP data into Delta
2. `02_Lakebase_Setup` → provision Lakebase and load data
3. `04_App_Deployment` → deploy the app (re-syncs source to `apps/` automatically)